# Intelligent Campus Cyber Security Threat Detection System
### Supervised Machine Learning Pipeline for Campus Network Intrusion Defense
---
**Author:** Campus Cyber Defense Research Lab  
**Methods:** Supervised Machine Learning (Random Forest, GBDT, MLP Neural Net, Decision Tree, Logistic Regression)  
**Subnets Monitored:** Student Dorms, CS Labs, Administration, University Server Farm, Guest Wi-Fi

## 1. Imports and Setup

In [ ]:
import sys
from pathlib import Path
import pandas as pd
import numpy as np

PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.dataset_generator import CampusDatasetGenerator
from src.preprocessor import CampusThreatPreprocessor
from src.models import get_model_registry
from src.evaluator import ModelEvaluator
from src.detector import CampusThreatDetector
from src.incident_advisor import IncidentAdvisor

print("Libraries and project modules loaded successfully!")

## 2. Generate and Inspect Synthetic Campus Flow Dataset
We model 7 classes: Benign, DDoS Flood, Brute Force, Port Scan, Web Attack (SQLi/XSS), Botnet C2, and Data Exfiltration.

In [ ]:
generator = CampusDatasetGenerator(random_state=42)
df = generator.generate(total_samples=3000)
print(f"Generated dataset shape: {df.shape}")
print("\nClass Distribution:")
print(df["label"].value_counts())
df.head()

## 3. Data Preprocessing & Feature Engineering

In [ ]:
from sklearn.model_selection import train_test_split

train_df, test_df = train_test_split(df, test_size=0.25, random_state=42, stratify=df["label"])
preprocessor = CampusThreatPreprocessor()
X_train, y_train = preprocessor.fit_transform(train_df, target_col="label")
X_test = preprocessor.transform(test_df)
y_test = preprocessor.label_encoder.transform(test_df["label"])

print(f"Training set samples: {X_train.shape[0]} | Features count: {X_train.shape[1]}")
print(f"Testing set samples: {X_test.shape[0]}")

## 4. Multi-Model Benchmark Comparison

In [ ]:
models = get_model_registry()
classes = list(preprocessor.label_encoder.classes_)

print(f"{'MODEL':<22} | {'ACCURACY':<10} | {'PRECISION':<10} | {'RECALL':<10} | {'F1-MACRO':<10}")
print('-' * 72)
for name, model in models.items():
    model.fit(X_train, y_train)
    metrics = ModelEvaluator.evaluate(model, X_test, y_test, target_names=classes, feature_names=preprocessor.feature_columns)
    acc = metrics['accuracy']
    prec = metrics['precision_macro']
    rec = metrics['recall_macro']
    f1 = metrics['f1_macro']
    print(f"{name:<22} | {acc:<10.4f} | {prec:<10.4f} | {rec:<10.4f} | {f1:<10.4f}")

## 5. Live Threat Detection and Automated Containment

In [ ]:
detector = CampusThreatDetector(models_dir=PROJECT_ROOT / 'models' / 'saved')
event = detector.simulate_stream_event()

print('=' * 65)
print(f"Flow: {event['src_ip']} ({event['flow_data']['src_subnet']}) -> {event['dst_ip']} ({event['flow_data']['dst_subnet']})")
print(f"Verdict: {event['prediction']} (Confidence: {event['confidence']*100:.1f}%)")
print(f"Severity: {event['advisory']['severity']}")
print(f"MITRE Technique: {event['advisory']['mitre_id']} - {event['advisory']['mitre_name']}")
print(f"Recommended Action: {event['advisory']['recommended_action']}")
print('=' * 65)
print('Generated Firewall Rule:')
print(event['advisory']['firewall_rule'])